In [1]:
## 01_Kumeleme

In [2]:
import pandas as pd
import numpy as np

veri = pd.read_csv("../data/clean/countries_clean.csv")
ulkeler = pd.read_csv("../data/clean/ulkeler_1m.csv")

# 161 büyük ülke, yalnızca BM tahmin yılları (1950–2023)
d = veri[veri.ISO3_code.isin(ulkeler.iso3) & (veri.Time <= 2023)].copy()
d = d[["ISO3_code", "Location", "Region", "Time", "TPopulation1July", "TFR", "LEx", "MedianAgePop", "IMR", "CNMR"]]
d["logIMR"] = np.log(d["IMR"])

print(d.shape, "boş değer:", d.isna().sum().sum())

(11914, 11) boş değer: 0


In [3]:
ozellikler = ["TFR", "LEx", "MedianAgePop", "logIMR", "CNMR"]
d[ozellikler].describe().round(2)

,TFR,LEx,MedianAgePop,logIMR,CNMR
count,11914.00,11914.00,11914.00,11914.00,11914.00
mean,4.17,61.99,23.25,3.65,0.41
std,2.08,12.74,7.79,1.17,15.70
min,0.71,10.99,13.19,0.30,-709.90
25%,2.18,52.51,16.98,2.83,-2.62
50%,3.90,64.41,19.96,3.90,-0.20
75%,6.16,71.86,28.87,4.64,1.91
max,8.86,85.51,48.96,6.05,357.14


In [4]:
d[ozellikler].corr().round(2)

,TFR,LEx,MedianAgePop,logIMR,CNMR
TFR,1.00,-0.85,-0.85,0.85,0.00
LEx,-0.85,1.00,0.73,-0.91,0.08
MedianAgePop,-0.85,0.73,1.00,-0.85,0.05
logIMR,0.85,-0.91,-0.85,1.00,-0.07
CNMR,0.00,0.08,0.05,-0.07,1.00


In [5]:
print(d.nsmallest(5, "LEx")[["Location", "Time", "LEx"]])
print(d.loc[d.CNMR.abs().sort_values(ascending=False).index[:8], ["Location", "Time", "CNMR"]])

          Location  Time      LEx
2318   South Sudan  1988  10.9891
12339     Cambodia  1977  11.2952
12340     Cambodia  1978  11.5726
12338     Cambodia  1976  11.6323
1868        Rwanda  1994  12.1575
                   Location  Time     CNMR
15088                Kuwait  1990 -709.899
9745   China, Hong Kong SAR  1967 -359.282
9742   China, Hong Kong SAR  1964  357.140
15089                Kuwait  1991  340.853
7488                Liberia  1990 -320.437
13577           Timor-Leste  1999 -315.012
1868                 Rwanda  1994 -305.949
15560                 Qatar  2006  209.940


In [6]:
alt, ust = d.CNMR.quantile([0.01, 0.99])
d["CNMR_k"] = d.CNMR.clip(alt, ust)
print(round(alt, 2), round(ust, 2))

-25.75 45.38


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def k_tara(ozellikler):
    X = StandardScaler().fit_transform(d[ozellikler])
    sonuc = []
    for k in range(3, 9):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        s = silhouette_score(X, km.labels_, sample_size=5000, random_state=42)
        sonuc.append({"k": k, "silhouette": round(s, 3), "inertia": round(km.inertia_)})
    return pd.DataFrame(sonuc)

set_A = ["TFR", "LEx", "MedianAgePop", "logIMR", "CNMR_k"]
set_B = ["TFR", "LEx", "MedianAgePop", "CNMR_k"]
set_C = ["TFR", "LEx", "MedianAgePop"]

for ad, s in [("A", set_A), ("B", set_B), ("C", set_C)]:
    print(ad, s)
    print(k_tara(s))
    print()

A ['TFR', 'LEx', 'MedianAgePop', 'logIMR', 'CNMR_k']
   k  silhouette  inertia
0  3       0.364    21205
1  4       0.395    15904
2  5       0.338    13219
3  6       0.292    11936
4  7       0.302    10898
5  8       0.308     9964

B ['TFR', 'LEx', 'MedianAgePop', 'CNMR_k']
   k  silhouette  inertia
0  3       0.501    17484
1  4       0.394    13059
2  5       0.328    11090
3  6       0.338     9971
4  7       0.292     9075
5  8       0.304     8263

C ['TFR', 'LEx', 'MedianAgePop']
   k  silhouette  inertia
0  3       0.478     6847
1  4       0.425     4852
2  5       0.382     3905
3  6       0.360     3271
4  7       0.332     2921
5  8       0.309     2655



In [8]:
import joblib

OZELLIKLER = ["TFR", "LEx", "MedianAgePop"]
olcekleyici = StandardScaler().fit(d[OZELLIKLER])
X = olcekleyici.transform(d[OZELLIKLER])
model = KMeans(n_clusters=5, random_state=42, n_init=50).fit(X)

# Etiketleri sabitle: ortalama doğurganlığı en yüksek küme = 1
d["ham"] = model.labels_
sira = d.groupby("ham").TFR.mean().sort_values(ascending=False).index
etiket = {int(ham): i + 1 for i, ham in enumerate(sira)}
d["kume"] = d.ham.map(etiket)

ISIMLER = {1: "Geçiş öncesi", 2: "Erken geçiş", 3: "Orta geçiş",
           4: "Geç geçiş", 5: "Geçiş sonrası"}
d["kume_adi"] = d.kume.map(ISIMLER)

merkez = d.groupby(["kume", "kume_adi"])[OZELLIKLER].mean().round(2)
merkez["gozlem"] = d.kume.value_counts().sort_index().values
merkez

,,TFR,LEx,MedianAgePop,gozlem
kume,kume_adi,,,,
1,Geçiş öncesi,6.71,42.30,17.27,2362
2,Erken geçiş,6.01,56.55,16.72,2668
3,Orta geçiş,3.71,65.16,19.89,2291
4,Geç geçiş,2.31,71.62,28.29,2843
5,Geçiş sonrası,1.54,77.10,37.48,1750


In [9]:
print("1990:", d[d.Time == 1990].kume.value_counts().sort_index().to_dict())
print("2023:", d[d.Time == 2023].kume.value_counts().sort_index().to_dict())
tr = d[d.ISO3_code == "TUR"].set_index("Time").kume
print("Türkiye basamak değişimleri:", tr[tr.diff() != 0].to_dict())

1990: {1: 22, 2: 39, 3: 40, 4: 38, 5: 22}
2023: {2: 8, 3: 44, 4: 42, 5: 67}
Türkiye basamak değişimleri: {1950: 1, 1959: 2, 1977: 3, 1997: 4, 2022: 5}


In [10]:
joblib.dump({"ozellikler": OZELLIKLER, "olcekleyici": olcekleyici, "model": model,
             "etiket": etiket, "isimler": ISIMLER}, "../models/kume_modeli.joblib")

# Tüm ülke-yıl atamaları (Sankey, harita ve pano bunu kullanacak)
d[["ISO3_code", "Location", "Region", "Time", "kume", "kume_adi", "TFR", "LEx", "MedianAgePop"]].rename(columns={
    "ISO3_code": "iso3", "Location": "country", "Region": "region", "Time": "year",
    "kume": "cluster", "kume_adi": "cluster_name", "TFR": "tfr", "LEx": "lex",
    "MedianAgePop": "median_age"}).to_csv("../data/out/kume_atamalari.csv", index=False)

# Kümelerin profili
merkez.reset_index().rename(columns={
    "kume": "cluster", "kume_adi": "cluster_name", "TFR": "tfr", "LEx": "lex",
    "MedianAgePop": "median_age", "gozlem": "n_obs"}).to_csv("../data/out/g02_kume_merkezleri.csv", index=False)

# Her yıl her basamakta kaç ülke ve kaç insan var
dag = d.groupby(["Time", "kume", "kume_adi"]).agg(
    n_countries=("ISO3_code", "size"),
    pop_millions=("TPopulation1July", lambda s: round(s.sum() / 1000, 2))).reset_index()
dag.columns = ["year", "cluster", "cluster_name", "n_countries", "pop_millions"]
dag.to_csv("../data/out/g02_kume_dagilimi.csv", index=False)

# Harita için 1990 ve 2023 kümeleri
h = d[d.Time.isin([1990, 2023])].pivot(index=["ISO3_code", "Location"], columns="Time", values="kume").reset_index()
h.columns = ["iso3", "country", "cluster_1990", "cluster_2023"]
h.to_csv("../data/out/g02_kume_harita.csv", index=False)
print("Kaydedildi.")

Kaydedildi.
